# Heretic Colab Setup
このノートブックは Heretic を Google Colab 上で動かします。GPU を選択し、必要な依存をインストールしてからモデルを実行します。

In [ ]:
#@title Optional: Mount Google Drive (チェックポイント保存のみ)
# Drive に依存したくない場合は False にしてください（デフォルト: False）。
MOUNT_DRIVE = False  #@param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    import os
    print('Mounting Google Drive...')
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/heretic'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print('Drive mounted at /content/drive. Using', DRIVE_ROOT)
else:
    print('Google Drive mounting skipped. Checkpoints will default to /content/checkpoints unless you change settings below.')


In [1]:
# ランタイム確認（GPU が利用可能か確認し、なければ対処法を表示します）
import shutil, sys, textwrap
if shutil.which('nvidia-smi') is None:
    print(textwrap.dedent('''
nvidia-smi が見つかりません。GPU が有効なランタイムを選択してください。
'''))
else:
    # GPU ドライバ情報を表示
    !nvidia-smi

Sat Apr 18 07:39:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# pip を最新化して依存を入れる（CUDA バージョンに合わせて index-url を変更してください）
!pip install pip -q
!pip install uv -q

!uv self version

uv 0.11.7 (x86_64-unknown-linux-gnu)


リポジトリをクローンします。

In [ ]:
#@title Clone Heretic (フォームで Repo/Branch を指定)
repo = "Akikukeo1/Live-Vision-Narrator"  #@param {type:"string"}
branch = "main"  #@param {type:"string"}
dest = "/content/Live-Vision-Narrator"  #@param {type:"string"}

!git clone --depth 1 --branch {branch} https://github.com/{repo}.git {dest}

# heretic サブディレクトリを指定
HERETIC_PATH = f"{dest}/heretic"
print(f'✓ HERETIC_PATH set to {HERETIC_PATH}')


依存をインストール・アップデートする必要があります。

In [ ]:
#@title Install heretic via uv (必須)
# NOTE: Clone セルを先に実行してください。

!cd {HERETIC_PATH} && uv venv --clear && uv pip install -e .

print('\n✓ Heretic installed successfully.')

# すべての依存を CUDA 128 向けに統一
print('\nSyncing all dependencies to CUDA 128...')
!uv pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache
!uv pip install --upgrade transformers --no-cache
!uv pip install --upgrade bitsandbytes --no-cache

print('\n✓ All dependencies installed and upgraded successfully.')


In [ ]:
# チェックポイント保存先の設定（Drive をマウントしている場合は Drive に、そうでなければ /content に保存）
import os, shutil
# MOUNT_DRIVE が定義されていなければ False を仮定
try:
    MOUNT_DRIVE
except NameError:
    MOUNT_DRIVE = False

if MOUNT_DRIVE:
    CHECKPOINT_DIR = '/content/drive/MyDrive/heretic/checkpoints'
else:
    CHECKPOINT_DIR = '/content/checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be saved to', CHECKPOINT_DIR)

# 簡単な同期例（必要に応じて手動で実行）
print('To sync checkpoints to Drive later (if MOUNT_DRIVE is True):')
print('  !rsync -av --progress /content/checkpoints/ /content/drive/MyDrive/heretic/checkpoints/')
print('Or use copy:')
print('  !cp -r /content/checkpoints/* /content/drive/MyDrive/heretic/checkpoints/')


In [ ]:
#@title Debug: Venv と CUDA ネイティブライブラリの確認
# 仮想環境内で check.py を実行し、bitsandbytes とネイティブライブラリの状況を確認します

!echo "== nvidia-smi =="
!nvidia-smi || true

!echo "\n== ldconfig libnvJitLink =="
!ldconfig -p | grep libnvJitLink || true

!echo "\n== .venv python check.py =="
!cd {HERETIC_PATH} && .venv/bin/python check.py || true

!echo "\n== .venv pip freeze =="
!cd {HERETIC_PATH} && .venv/bin/pip freeze | sed -n '1,200p' || true

!echo "\n== bitsandbytes import test (.venv) =="
!cd {HERETIC_PATH} && .venv/bin/python -c "import importlib, sys
try:
    import bitsandbytes as bnb
    print('bitsandbytes import OK', getattr(bnb, '__file__', ''))
except Exception as e:
    print('bitsandbytes import FAILED:', e)
    import traceback; traceback.print_exc()


In [ ]:
#@title Install pexpect for auto-response
# Auto-resume 機能用に pexpect をインストール
!uv pip install pexpect -q
print('✓ pexpect installed successfully')

## モデルのセットアップと実行
HuggingFace からモデルをダウンロードして、Heretic で実行します。


In [ ]:
#@title Download model from HuggingFace
from huggingface_hub import snapshot_download
import os

model_id = "google/gemma-4-E4B-it"  #@param {type:"string"}
model_dir = "/content/models"  #@param {type:"string"}

os.makedirs(model_dir, exist_ok=True)

print(f'Downloading {model_id} to {model_dir}...')
model_path = snapshot_download(
    repo_id=model_id,
    cache_dir=model_dir,
    local_dir=f"{model_dir}/{model_id.replace('/', '--')}",
    local_dir_use_symlinks=False
)

print(f'✓ Model downloaded to: {model_path}')
MODEL_PATH = model_path


In [ ]:
#@title Check storage usage
import os
import subprocess

# ストレージ容量を確認
result = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
print('Storage usage:')
print(result.stdout)

# モデルのサイズを確認
if 'MODEL_PATH' in globals() and os.path.exists(MODEL_PATH):
    result = subprocess.run(['du', '-sh', MODEL_PATH], capture_output=True, text=True)
    print(f'\nModel size: {result.stdout}')


In [ ]:
#@title Restore from Drive & Start Background Sync (チェックポイント自動同期)
import os
import subprocess
import threading
import time

# ローカル保存先（絶対パス）
LOCAL_CKPTS = '/content/checkpoints'
LOCAL_MODELS = '/content/models'
os.makedirs(LOCAL_CKPTS, exist_ok=True)
os.makedirs(LOCAL_MODELS, exist_ok=True)

# Drive 側の保存先
DRIVE_CKPTS = '/content/drive/MyDrive/heretic/checkpoints'
DRIVE_MODELS = '/content/drive/MyDrive/heretic/models'

# 既に Drive マウント済みと仮定（前のセルで実施済み）
# 必要に応じてここで drive.mount() を実行してください

print("=== Restoring checkpoints from Drive ===")
os.makedirs(DRIVE_CKPTS, exist_ok=True)
subprocess.run(['rsync', '-av', '--progress', f'{DRIVE_CKPTS}/', f'{LOCAL_CKPTS}/'], check=False)

print("=== Starting background sync thread ===")
SYNC_INTERVAL = 60  # 60 秒ごとに同期

def sync_loop():
    while True:
        try:
            subprocess.run(['rsync', '-av', '--progress', f'{LOCAL_CKPTS}/', f'{DRIVE_CKPTS}/'], check=True, timeout=120)
            print(f"[sync] Checkpoints synced to Drive at {time.strftime('%Y-%m-%d %H:%M:%S')}")
        except Exception as e:
            print(f"[sync error] {e}")
        time.sleep(SYNC_INTERVAL)

# デーモンスレッドで同期を開始
sync_thread = threading.Thread(target=sync_loop, daemon=True, name='checkpoint_sync')
sync_thread.start()
print(f"Background sync started (interval: {SYNC_INTERVAL}s)")
print(f"Local checkpoints: {LOCAL_CKPTS}")
print(f"Drive checkpoints: {DRIVE_CKPTS}")

In [ ]:
#@title Run Heretic (Auto-Resume with pexpect)
# NOTE: モデルダウンロードセルと「Restore from Drive & Start Background Sync」セルを先に実行してください

import subprocess
import os
import pexpect

use_quantization = True  #@param {type:"boolean"}
skip_batch_autodetect = True  #@param {type:"boolean"}
fixed_batch_size = 128  #@param {type:"integer", min:1, max:512}
AUTO_RESUME = True  #@param {type:"boolean"}
AUTO_RESUME_TIMEOUT = 300  #@param {type:"integer", min:10, max:3600}
trial_selection = 1  #@param {type:"integer", min:1, max:100}
auto_exit_after_trial = True  #@param {type:"boolean"}

LOCAL_CKPTS = '/content/checkpoints'
DRIVE_CKPTS = '/content/drive/MyDrive/heretic/checkpoints'

# コマンド構築
cmd = f"cd {HERETIC_PATH} && uv run --no-sync heretic --model {MODEL_PATH} --study-checkpoint-dir {LOCAL_CKPTS}"
if use_quantization:
    cmd += " --quantization bnb_4bit"
if skip_batch_autodetect:
    cmd += f" --batch-size {fixed_batch_size}"

log_file = f"{HERETIC_PATH}/heretic_run.log"

print(f"Running command: {cmd}")
print(f"Checkpoints saved to: {LOCAL_CKPTS}")
print(f"Log file: {log_file}")
print(f"Auto-resume enabled: {AUTO_RESUME}")
print(f"Auto-select trial: {trial_selection}")
print(f"Auto-exit after trial: {auto_exit_after_trial}")
print()

# Auto-resume & Auto-select with pexpect
if AUTO_RESUME:
    print("=== Starting Heretic with auto-resume/select (pexpect) ===")
    try:
        with open(log_file, 'a', encoding='utf-8') as log:
            child = pexpect.spawn('/bin/bash', ['-lc', cmd], encoding='utf-8', timeout=None)
            child.logfile_read = log

            # 1. 開始時のプロンプト（Continue / Restart）を待つ
            try:
                child.expect(r'How would you like to proceed\?', timeout=AUTO_RESUME_TIMEOUT)
                print("[auto-resume] Detected continuation prompt, sending '1' (Continue)...")
                child.sendline('1')
            except (pexpect.TIMEOUT, pexpect.EOF):
                print("[auto-resume] Initial prompt skipped (Normal start).")

            # 2. 最適化完了後の「Which trial do you want to use?」を待つ
            try:
                # 非常に長い時間がかかる可能性があるため timeout=None
                child.expect(r'Which trial do you want to use\?', timeout=None)
                print(f"[auto-select] Optimization finished! Selecting Trial [{trial_selection}]...")
                child.sendline(str(trial_selection))

                # 3. 保存やチャットのメニューが出た場合の処理
                if auto_exit_after_trial:
                    try:
                        child.expect(r'Enter number:', timeout=60)
                        print("[auto-select] Menu detected. Exiting and saving...")
                        child.sendline('12')  # [12] Exit program
                    except (pexpect.TIMEOUT, pexpect.EOF):
                        print("[auto-select] Menu not found or process already ended.")

            except pexpect.TIMEOUT:
                print("[auto-select] Timeout waiting for trial selection menu.")
            except pexpect.EOF:
                print("[auto-select] Process ended before selection menu.")

            # プロセス終了を待つ
            try:
                child.expect(pexpect.EOF)
            except:
                pass

            result_code = child.exitstatus if child.exitstatus is not None else 0
    except Exception as e:
        print(f"[auto-resume error] {e}. Falling back to subprocess.run...")
        result = subprocess.run(
            f"{cmd} 2>&1 | tee {log_file}",
            shell=True,
            check=False
        )
        result_code = result.returncode
else:
    print("=== Starting Heretic with standard subprocess ===")
    result = subprocess.run(
        f"{cmd} 2>&1 | tee {log_file}",
        shell=True,
        check=False
    )
    result_code = result.returncode

print()
print("=== Final sync to Drive ===")
os.makedirs(DRIVE_CKPTS, exist_ok=True)
subprocess.run(['rsync', '-av', '--progress', f'{LOCAL_CKPTS}/', f'{DRIVE_CKPTS}/'], check=False)
print(f"Final sync completed. Checkpoints saved to Drive: {DRIVE_CKPTS}")
print()
print(f"Exit code: {result_code}")
if result_code == 0:
    print("✓ Heretic completed successfully")
else:
    print("✗ Heretic exited with error (see log above)")